<a href="https://colab.research.google.com/github/mobius29er/AIML_Class/blob/main/try_it_20_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Comparing Aggregate Models for Regression

This try-it focuses on utilizing ensemble models in a regression setting.  Much like you have used individual classification estimators to form an ensemble of estimators -- here your goal is to explore ensembles for regression models.  As with your earlier assignment, you will use scikitlearn to carry out the ensembles using the `VotingRegressor`.   


#### Dataset and Task

Below, a dataset containing census information on individuals and their hourly wage is loaded using the `fetch_openml` function.  OpenML is another repository for datasets [here](https://www.openml.org/).  Your task is to use ensemble methods to explore predicting the `wage` column of the data.  Your ensemble should at the very least consider the following models:

- `LinearRegression` -- perhaps you even want the `TransformedTargetRegressor` here.
- `KNeighborsRegressor`
- `DecisionTreeRegressor`
- `Ridge`
- `SVR`

Tune the `VotingRegressor` to try to optimize the prediction performance and determine if the wisdom of the crowd performed better in this setting than any of the individual models themselves.  Report back on your findings and discuss the interpretability of your findings.  Is there a way to determine what features mattered in predicting wages?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_openml

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance

# Load dataset
survey = fetch_openml(data_id=534, as_frame=True).frame

In [3]:
# Display the first few rows to understand the data
survey.head()

,EDUCATION,SOUTH,SEX,EXPERIENCE,UNION,WAGE,AGE,RACE,OCCUPATION,SECTOR,MARR
0,8,no,female,21,not_member,5.10,35,Hispanic,Other,Manufacturing,Married
1,9,no,female,42,not_member,4.95,57,White,Other,Manufacturing,Married
2,12,no,male,1,not_member,6.67,19,White,Other,Manufacturing,Unmarried
3,12,no,male,4,not_member,4.00,22,White,Other,Other,Unmarried
4,12,no,male,17,not_member,7.50,35,White,Other,Other,Married


# Task
Analyze the provided dataset to predict hourly wage using ensemble methods, specifically a `VotingRegressor` with `LinearRegression`, `KNeighborsRegressor`, `DecisionTreeRegressor`, `Ridge`, and `SVR` as base models. Tune the `VotingRegressor` using `GridSearchCV`, compare its performance to the individual models, and discuss feature importance and interpretability.

## Data preprocessing

### Subtask:
Handle categorical features, split data into training and testing sets, and scale numerical features.


**Reasoning**:
Identify categorical features, apply one-hot encoding, split data, identify numerical features, and scale numerical features as per the instructions.



In [4]:
# Identify categorical columns
categorical_cols = survey.select_dtypes(include='category').columns

# Apply one-hot encoding
survey_encoded = pd.get_dummies(survey, columns=categorical_cols, drop_first=True)

# Separate target variable from features
X = survey_encoded.drop('WAGE', axis=1)
y = survey_encoded['WAGE']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numerical columns after one-hot encoding
numerical_cols_encoded = X_train.select_dtypes(include=np.number).columns

# Apply standard scaling to numerical columns
scaler = StandardScaler()
X_train[numerical_cols_encoded] = scaler.fit_transform(X_train[numerical_cols_encoded])
X_test[numerical_cols_encoded] = scaler.transform(X_test[numerical_cols_encoded])

display(X_train.head())
display(X_test.head())
display(y_train.head())
display(y_test.head())

,EDUCATION,EXPERIENCE,AGE,SOUTH_yes,SEX_male,UNION_not_member,RACE_Other,RACE_White,OCCUPATION_Management,OCCUPATION_Other,OCCUPATION_Professional,OCCUPATION_Sales,OCCUPATION_Service,SECTOR_Manufacturing,SECTOR_Other,MARR_Unmarried
489,1.138280,-0.173148,0.070028,False,False,True,False,True,False,False,True,False,False,False,True,False
299,-0.401057,0.069259,-0.014921,False,True,True,False,True,False,False,False,False,False,False,True,False
526,0.753446,-0.657962,-0.524613,False,True,True,False,True,False,False,True,False,False,False,True,True
513,-0.401057,1.685307,1.684052,False,True,False,False,True,False,False,True,False,False,False,True,False
312,-0.016223,-1.465986,-1.543996,True,False,True,False,True,False,False,False,False,False,False,True,True


,EDUCATION,EXPERIENCE,AGE,SOUTH_yes,SEX_male,UNION_not_member,RACE_Other,RACE_White,OCCUPATION_Management,OCCUPATION_Other,OCCUPATION_Professional,OCCUPATION_Sales,OCCUPATION_Service,SECTOR_Manufacturing,SECTOR_Other,MARR_Unmarried
222,-0.401057,2.170121,2.193744,False,False,True,False,True,False,False,False,True,False,False,True,False
131,-0.401057,-1.223579,-1.374099,False,True,True,False,True,False,True,False,False,False,True,False,True
149,-0.016223,-0.900369,-0.949356,True,True,True,False,True,False,True,False,False,False,True,False,False
244,-0.401057,0.150062,0.070028,False,True,True,False,True,False,False,False,True,False,False,True,False
84,-0.016223,-0.819567,-0.864407,False,True,False,True,False,False,True,False,False,False,True,False,True


,WAGE
489,14.00
299,7.69
526,12.50
513,15.00
312,4.50


,WAGE
222,6.4
131,5.5
149,6.0
244,5.5
84,9.0


## Define base models

### Subtask:
Instantiate the individual regression models to be used in the ensemble.


**Reasoning**:
Instantiate the required individual regression models with specified parameters for the ensemble.



In [5]:
# Instantiate individual models
lr = LinearRegression()
knn = KNeighborsRegressor(n_neighbors=5)
dt = DecisionTreeRegressor(random_state=42)
ridge = Ridge(alpha=1.0)
svr = SVR(kernel='rbf')

## Create voting regressor

### Subtask:
Set up the `VotingRegressor` with the defined base models.


**Reasoning**:
Set up the VotingRegressor with the defined base models.



In [6]:
# Create a list of base models
estimators = [
    ('lr', lr),
    ('knn', knn),
    ('dt', dt),
    ('ridge', ridge),
    ('svr', svr)
]

# Instantiate the VotingRegressor
voting_regressor = VotingRegressor(estimators=estimators)

## Tune ensemble model

### Subtask:
Use `GridSearchCV` to find optimal hyperparameters for the `VotingRegressor`.


**Reasoning**:
Define the parameter grid for GridSearchCV and instantiate GridSearchCV with the voting regressor and the parameter grid. Then, fit GridSearchCV to the training data and print the best parameters and the best score.



In [7]:
# Define the parameter grid for GridSearchCV
param_grid = {
    'weights': [[1, 1, 1, 1, 1], [2, 1, 1, 1, 1], [1, 2, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 2, 1], [1, 1, 1, 1, 2],
                [2, 2, 1, 1, 1], [1, 2, 2, 1, 1], [1, 1, 2, 2, 1], [1, 1, 1, 2, 2], [2, 1, 2, 1, 1], [1, 2, 1, 2, 1],
                [1, 1, 2, 1, 2], [2, 1, 1, 2, 1], [1, 2, 1, 1, 2], [2, 1, 1, 1, 2]]
}

# Instantiate GridSearchCV with the voting regressor and the parameter grid
grid_search = GridSearchCV(estimator=voting_regressor, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Print the best parameters and the corresponding best score
print("Best parameters: ", grid_search.best_params_)
print("Best negative mean squared error: ", grid_search.best_score_)

Best parameters:  {'weights': [2, 1, 1, 2, 1]}
Best negative mean squared error:  -19.62698254047063


## Evaluate model performance

### Subtask:
Evaluate the performance of the tuned `VotingRegressor` and the individual models using appropriate metrics.


**Reasoning**:
Retrieve the best estimator from the GridSearchCV object, make predictions on the test set, calculate the MSE for the VotingRegressor, train and predict with individual models, calculate MSE for each individual model, and print all MSEs.



In [8]:
# Retrieve the best estimator
best_voting_regressor = grid_search.best_estimator_

# Make predictions with the best VotingRegressor
voting_regressor_pred = best_voting_regressor.predict(X_test)

# Calculate MSE for the VotingRegressor
voting_regressor_mse = mean_squared_error(y_test, voting_regressor_pred)

# Train and evaluate individual models
individual_models = {
    'Linear Regression': lr,
    'KNeighbors Regressor': knn,
    'Decision Tree Regressor': dt,
    'Ridge': ridge,
    'SVR': svr
}

individual_mse = {}

for name, model in individual_models.items():
    # Train the individual model
    model.fit(X_train, y_train)
    # Make predictions
    pred = model.predict(X_test)
    # Calculate MSE
    mse = mean_squared_error(y_test, pred)
    individual_mse[name] = mse

# Print the MSEs
print(f"Voting Regressor MSE: {voting_regressor_mse}")
for name, mse in individual_mse.items():
    print(f"{name} MSE: {mse}")

Voting Regressor MSE: 20.17284706576886
Linear Regression MSE: 19.50259732411242
KNeighbors Regressor MSE: 19.840783813084116
Decision Tree Regressor MSE: 58.53001542056076
Ridge MSE: 19.492914677916787
SVR MSE: 20.859540593601636


## Feature importance analysis

### Subtask:
Explore methods to determine feature importance for the ensemble model.


**Reasoning**:
Calculate permutation importance for the best VotingRegressor to understand feature importance.



In [9]:
# Calculate permutation importance for the best VotingRegressor
result = permutation_importance(best_voting_regressor, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

# Store the results
sorted_idx = result.importances_mean.argsort()
permutation_importance_df = pd.DataFrame({
    'feature': X_test.columns[sorted_idx],
    'importance_mean': result.importances_mean[sorted_idx],
    'importance_std': result.importances_std[sorted_idx]
})

# Print the feature importance scores
print("Permutation Importance:")
print(permutation_importance_df)

Permutation Importance:
                    feature  importance_mean  importance_std
0          UNION_not_member        -0.011389        0.017237
1          OCCUPATION_Other        -0.003130        0.004088
2              SECTOR_Other        -0.002049        0.006878
3                RACE_Other        -0.000444        0.002005
4            MARR_Unmarried        -0.000279        0.006357
5          OCCUPATION_Sales         0.003713        0.004104
6      SECTOR_Manufacturing         0.003956        0.007569
7                RACE_White         0.004009        0.005248
8                 SOUTH_yes         0.004656        0.014583
9        OCCUPATION_Service         0.011005        0.014554
10                      AGE         0.040386        0.020257
11    OCCUPATION_Management         0.050329        0.017184
12  OCCUPATION_Professional         0.062193        0.015896
13                 SEX_male         0.069330        0.032081
14               EXPERIENCE         0.085209        0.042172


## Report findings

### Subtask:
Summarize the results, compare the performance of the ensemble to individual models, and discuss feature importance.


**Reasoning**:
Summarize the results, compare the performance of the ensemble to individual models, and discuss feature importance and interpretability based on the previously obtained MSE values and permutation importance results.



In [10]:
print("--- Model Performance Comparison (MSE) ---")
print(f"Voting Regressor: {voting_regressor_mse:.4f}")
for name, mse in individual_mse.items():
    print(f"{name}: {mse:.4f}")

print("\n--- Feature Importance (Permutation Importance) ---")
print(permutation_importance_df)

print("\n--- Summary and Discussion ---")
print("The Voting Regressor achieved an MSE of {:.4f} on the test set.".format(voting_regressor_mse))
print("Comparing this to the individual models:")
for name, mse in individual_mse.items():
    print(f"- {name}: MSE = {mse:.4f}")

print("\nIn this case, the individual Ridge and Linear Regression models performed slightly better than the tuned Voting Regressor in terms of MSE.")
print("The Decision Tree Regressor had significantly worse performance compared to the ensemble and other individual models.")
print("The KNeighbors Regressor and SVR performed similarly to the Voting Regressor.")

print("\nRegarding feature importance, the permutation importance analysis for the tuned Voting Regressor indicates the following:")
print("The most important features for predicting wage appear to be:")
print(permutation_importance_df.tail(5))
print("\nThe least important features appear to be:")
print(permutation_importance_df.head(5))
print("\nEDUCATION, EXPERIENCE, and SEX_male are among the top features influencing wage predictions according to this analysis.")
print("Features like UNION_not_member, OCCUPATION_Other, and SECTOR_Other seem to have less impact.")
print("The interpretability of these findings comes from understanding how much the model's prediction error increases when a feature's information is removed (permuted).")
print("Higher importance values suggest that the feature is a stronger predictor of wage in this ensemble model.")

--- Model Performance Comparison (MSE) ---
Voting Regressor: 20.1728
Linear Regression: 19.5026
KNeighbors Regressor: 19.8408
Decision Tree Regressor: 58.5300
Ridge: 19.4929
SVR: 20.8595

--- Feature Importance (Permutation Importance) ---
                    feature  importance_mean  importance_std
0          UNION_not_member        -0.011389        0.017237
1          OCCUPATION_Other        -0.003130        0.004088
2              SECTOR_Other        -0.002049        0.006878
3                RACE_Other        -0.000444        0.002005
4            MARR_Unmarried        -0.000279        0.006357
5          OCCUPATION_Sales         0.003713        0.004104
6      SECTOR_Manufacturing         0.003956        0.007569
7                RACE_White         0.004009        0.005248
8                 SOUTH_yes         0.004656        0.014583
9        OCCUPATION_Service         0.011005        0.014554
10                      AGE         0.040386        0.020257
11    OCCUPATION_Management 

## Summary:

### Data Analysis Key Findings

*   The tuned `VotingRegressor` achieved a Mean Squared Error (MSE) of 20.1728 on the test set.
*   The individual `Ridge` (MSE: 19.4929) and `LinearRegression` (MSE: 19.5026) models performed slightly better than the tuned `VotingRegressor`.
*   The `DecisionTreeRegressor` had significantly worse performance (MSE: 58.5300) compared to the ensemble and other individual models.
*   The `KNeighborsRegressor` (MSE: 19.8408) and `SVR` (MSE: 20.8595) performed similarly to the tuned `VotingRegressor`.
*   Permutation importance analysis for the tuned `VotingRegressor` identified 'EDUCATION' (0.3686), 'EXPERIENCE' (0.0852), and 'SEX\_male' (0.0693) as the most important features for predicting wage.
*   Features like 'UNION\_not\_member' (-0.0114), 'OCCUPATION\_Other' (-0.0031), and 'SECTOR\_Other' (-0.0020) appeared to have less impact according to the permutation importance analysis.

### Insights or Next Steps

*   Although the ensemble did not outperform the best individual models in this instance, exploring different ensemble techniques (e.g., Bagging, Boosting) or different base models could potentially yield better results.
*   Further investigation into the features identified as most important (EDUCATION, EXPERIENCE, SEX\_male) could provide deeper insights into wage determinants and potentially inform feature engineering or data collection efforts.
